# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AlaaSherif-Ibrahim/FLYRANK_AI/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane locked: Lane 2 — Refresh / Content Opportunity Scoring.** One notebook, three jobs:
check two signals first, encode one transparent rule into a ranked queue
(`work/outputs/baseline_action_score.csv`), then read my own top ten with a skeptic's eye.
Data: the starter slice (`data/raw/content_refresh_anonymized.csv`, 30k pages) — the same slice
and label the Week-5 model will be measured against.

Evaluation label throughout: `is_declining_label` (1 = page is in the declining bucket). It is
derived from `trend_direction`/`trend_pct`, so those two columns are **evaluation-only** here —
the card's hard constraint is that no label-derived input touches the rule.

In [1]:
# Setup — get the repo and data, wherever this kernel starts.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AlaaSherif-Ibrahim/FLYRANK_AI"
REPO_DIR = "FLYRANK_AI"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "scikit-learn"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found - are you at the repo root?"
print("Starter data found. Ready.")

Working dir: C:\Users\alaa\FLYRANK_AI
Starter data found. Ready.


## 1. My rule and its reason codes

### Signal checks first — two signals the rule idea leans on

Both are **flag-linked** signals from the session: staleness sits behind FlyRank's refresh
flags, and CTR-vs-position sits behind the CTR-fix logic. Each gets a bucket table with n
printed and a one-word verdict: CONFIRMED, OPPOSITE, MIXED or FALSE. A clearly-explained
negative counts as a win — it just saved the rule from leaning on a myth.

**Signal A — staleness (`days_since_last_update`) vs decline.** Expectation going in:
older content declines more. Verdict follows the table.

In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
y = df["is_declining_label"]
base_rate = y.mean()

bins = [-1, 90, 180, 10_000]
labs = ["0-90d fresh", "91-180d aging", "180d+ stale"]
df["stale_bucket"] = pd.cut(df["days_since_last_update"], bins=bins, labels=labs)
tA = df.groupby("stale_bucket", observed=True).agg(
    n=("is_declining_label", "size"),
    declining_n=("is_declining_label", "sum"),
    decline_pct=("is_declining_label", lambda s: round(100 * s.mean(), 1)),
)
print(tA.to_string())
print(f"\nbase rate for reference: {100 * base_rate:.1f}%")
print("VERDICT: MIXED - decline rises fresh->aging (51%->61%) but COLLAPSES in the stale bucket "
      "(47%), and that bucket holds only n=174 pages. This slice has almost no truly old content "
      "(97% of pages were updated within ~6 months), so staleness is directionally suggestive but "
      "too thin to lean a rule on. The rule below does NOT use it.")

                   n  declining_n  decline_pct
stale_bucket                                  
0-90d fresh    20655        10576         51.2
91-180d aging   9171         5604         61.1
180d+ stale      174           82         47.1

base rate for reference: 54.2%
VERDICT: MIXED - decline rises fresh->aging (51%->61%) but COLLAPSES in the stale bucket (47%), and that bucket holds only n=174 pages. This slice has almost no truly old content (97% of pages were updated within ~6 months), so staleness is directionally suggestive but too thin to lean a rule on. The rule below does NOT use it.


**Signal B — CTR within good positions vs decline** (behind the CTR-fix logic).
Expectation: among pages that already rank on page one and still get seen, weak click-through
marks a snippet/intent problem — and co-occurs with decline.

In [3]:
pool_b = df[(df["avg_position"] > 0) & (df["avg_position"] <= 10)
            & (df["impressions_90d"] >= 500)].copy()
qb = pd.qcut(pool_b["ctr"], q=3, duplicates="drop")
tB = pool_b.groupby(qb, observed=True).agg(
    n=("is_declining_label", "size"),
    declining_n=("is_declining_label", "sum"),
    decline_pct=("is_declining_label", lambda s: round(100 * s.mean(), 1)),
)
print(f"pool B (page-one ranked, avg_position 0-10, impressions_90d >= 500): n={len(pool_b):,}")
print(tB.to_string())
print("\n(ctr is x100 percentage points per the data dictionary; tercile edges at 0.15 / 0.36.)")
print("VERDICT: CONFIRMED - monotone 69.9% -> 58.0% -> 48.4% across rising CTR, a ~21.5pp spread, "
      "every bucket n > 2,400. Low CTR at good positions is real, measured signal on this slice. "
      "This is the state the CTR-fix logic flags - and the one my rule encodes.")

pool B (page-one ranked, avg_position 0-10, impressions_90d >= 500): n=7,564
                   n  declining_n  decline_pct
ctr                                           
(-0.001, 0.15]  2532         1770         69.9
(0.15, 0.36]    2578         1496         58.0
(0.36, 5.42]    2454         1188         48.4

(ctr is x100 percentage points per the data dictionary; tercile edges at 0.15 / 0.36.)
VERDICT: CONFIRMED - monotone 69.9% -> 58.0% -> 48.4% across rising CTR, a ~21.5pp spread, every bucket n > 2,400. Low CTR at good positions is real, measured signal on this slice. This is the state the CTR-fix logic flags - and the one my rule encodes.


### The rule, in plain words first

> *"Refresh-first the pages that already rank on page one and still get seen, but barely get
> clicked — and start with the ones where search demand is biggest."*

Three readable conditions, no fitted weights:

| Condition | Threshold | Why |
|---|---|---|
| ranked on page one | `0 < avg_position <= 10` | the page is seen; the problem isn't ranking |
| real visibility | `impressions_90d >= 500` | below that, CTR is noise |
| snippet underperforms | `ctr <= 0.15` | bottom tercile of Signal B — the confirmed state |

- **Score:** `search_volume` inside the rule pool — demand magnitude orders the queue (biggest
  audience served first). Transparent by construction.
- **ONE reason code:** `ranked_but_unclicked` — every queue row carries why it scored.
- **Action label:** `refresh_review` — hand to the editor for snippet/intent rework.

Leakage guard: the four inputs are trailing-90-day snapshot fields, knowable today;
`trend_direction`/`trend_pct` (the label's sources) touch nothing but evaluation.

In [4]:
RULE_INPUTS = ["avg_position", "impressions_90d", "ctr", "search_volume"]
FORBIDDEN = ["trend_direction", "trend_pct", "is_declining_label"]

assert not set(RULE_INPUTS) & set(FORBIDDEN), "label-derived column entered the rule"
assert all(c in df.columns for c in RULE_INPUTS), "rule input missing from the slice"
print("rule inputs:", RULE_INPUTS)
print("leakage guard holds:", not (set(RULE_INPUTS) & set(FORBIDDEN)))

rule inputs: ['avg_position', 'impressions_90d', 'ctr', 'search_volume']
leakage guard holds: True


## 2. Build the ranked queue (writes the CSV)

In [5]:
import json

from sklearn.dummy import DummyClassifier

pool_mask = ((df["avg_position"] > 0) & (df["avg_position"] <= 10)
             & (df["impressions_90d"] >= 500) & (df["ctr"] <= 0.15))
queue = df[pool_mask].copy()
queue["action_score"] = queue["search_volume"]
queue["reason_code"] = "ranked_but_unclicked"
queue["action_label"] = "refresh_review"
queue = queue.sort_values("action_score", ascending=False).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)

csv_path = os.path.join("work", "outputs", "baseline_action_score.csv")
os.makedirs(os.path.dirname(csv_path), exist_ok=True)
queue[["rank", "content_id", "client_id", "action_score", "reason_code", "action_label",
       "search_volume", "impressions_90d", "clicks_90d", "avg_position", "ctr",
       "days_since_last_update", "content_age_days"]].to_csv(csv_path, index=False)
print(f"wrote {csv_path} | {len(queue):,} rows")

def precision_at_k(labels, k):
    return float(np.asarray(labels)[:k].mean())

p50 = precision_at_k(queue["is_declining_label"], min(50, len(queue)))
p100 = precision_at_k(queue["is_declining_label"], min(100, len(queue)))

dummy = DummyClassifier(strategy="most_frequent").fit(df[["impressions_90d"]], y)
rng = np.random.default_rng(42)
dummy_p50 = np.mean([precision_at_k(y.sample(frac=1.0, random_state=s).to_numpy(), 50)
                     for s in range(200)])

metrics = {
    "lane": "2_refresh_scoring",
    "slice": "data/raw/content_refresh_anonymized.csv (30k pages)",
    "label": "is_declining_label",
    "rule_inputs": RULE_INPUTS,
    "thresholds": {"position": "(0, 10]", "impressions_90d": ">=500", "ctr": "<=0.15"},
    "pool_n": int(len(queue)),
    "pool_rate": round(float(queue["is_declining_label"].mean()), 3),
    "p_at_50": round(p50, 3),
    "p_at_100": round(p100, 3),
    "base_rate": round(float(base_rate), 3),
}
json_path = os.path.join("work", "outputs", "baseline_metrics.json")
with open(json_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"\nbase rate (random picks):        {base_rate:.3f}")
print(f"dummy majority baseline P@50:    {dummy_p50:.3f} (dummy predicts: {dummy.predict([[0]])[0]})")
print(f"HAND RULE  Precision@50 = {p50:.3f} -> {round(p50*50)} of top-50 actually declining")
print(f"HAND RULE  Precision@100 = {p100:.3f}")
print(f"reference points: shipped hand-rule 0.240, shipped random forest 0.740 (W02)")
print(f"\nmetrics receipt -> {json_path}")

wrote work\outputs\baseline_action_score.csv | 2,532 rows



base rate (random picks):        0.542
dummy majority baseline P@50:    0.542 (dummy predicts: 1)
HAND RULE  Precision@50 = 0.700 -> 35 of top-50 actually declining
HAND RULE  Precision@100 = 0.670
reference points: shipped hand-rule 0.240, shipped random forest 0.740 (W02)

metrics receipt -> work\outputs\baseline_metrics.json


## 3. Top-10 review

One line each: the action, why it's there, and what would make it wrong. The "wrong" notes are
generated from each row's own fields — threshold proximity, recent updates, SERP-feature squeeze,
measurement gaps, competition.

In [6]:
def wrong_if(r):
    notes = []
    if r["ctr"] >= 0.13:
        notes.append("ctr sits right at the 0.15 cut, threshold noise could flip this pick")
    if r["days_since_last_update"] <= 14:
        notes.append("updated within 14 days - may be mid-fix or still ramping")
    if 0 < r["avg_position"] <= 3:
        notes.append("top-3 slot yet low ctr - SERP features may squeeze clicks, not snippet quality")
    if pd.isna(r["word_count"]) or r["engagement_rate"] == 0:
        notes.append("no GA4 engagement and/or missing word count - possible measurement gap")
    if r["competition_level"] == "HIGH":
        notes.append("HIGH competition - demand-side squeeze a rewrite may not lift")
    return "; ".join(notes[:2]) if notes else "no red flag beyond the rule itself - check query mix manually"

for _, r in queue.head(10).iterrows():
    why = (f"pos {r['avg_position']:.1f} but ctr {r['ctr']:.2f}%, "
           f"{r['impressions_90d']:,} impr/90d, demand {r['search_volume']:,.0f}/mo")
    print(f"#{r['rank']:>2} {r['content_id']} [{r['action_label']}] because {r['reason_code']}: {why}")
    print(f"     label={r['is_declining_label']} | wrong if: {wrong_if(r)}")


# 1 content_f76ccf7a7834 [refresh_review] because ranked_but_unclicked: pos 9.5 but ctr 0.15%, 646 impr/90d, demand 49,500/mo
     label=1 | wrong if: ctr sits right at the 0.15 cut, threshold noise could flip this pick; no GA4 engagement and/or missing word count - possible measurement gap
# 2 content_6ef3dcb7be11 [refresh_review] because ranked_but_unclicked: pos 6.0 but ctr 0.07%, 1,380 impr/90d, demand 27,100/mo
     label=1 | wrong if: updated within 14 days - may be mid-fix or still ramping; no GA4 engagement and/or missing word count - possible measurement gap
# 3 content_5c5fab9d41e7 [refresh_review] because ranked_but_unclicked: pos 5.4 but ctr 0.06%, 1,794 impr/90d, demand 22,200/mo
     label=0 | wrong if: no GA4 engagement and/or missing word count - possible measurement gap
# 4 content_f854023b075b [refresh_review] because ranked_but_unclicked: pos 5.1 but ctr 0.00%, 627 impr/90d, demand 18,100/mo
     label=0 | wrong if: no GA4 engagement and/or missing word count - possi

## 4. Weak picks + leakage check

In [7]:
# The weakest pick in the top 10, read skeptically.
weak = queue.head(10).loc[lambda t: t["is_declining_label"] == 0].sort_values("rank")
print(f"top-10 precision: {queue.head(10)['is_declining_label'].mean():.1f} "
      f"({int(queue.head(10)['is_declining_label'].sum())} of 10 flagged declining)")
for _, r in weak.iterrows():
    print(f"\nWEAK #{r['rank']} {r['content_id']}: pos {r['avg_position']:.1f}, ctr {r['ctr']:.2f}%, "
          f"{r['impressions_90d']:,} impr/90d, updated {r['days_since_last_update']:.0f}d ago, "
          f"label=not declining")
    print(f"   why it looks wrong: {wrong_if(r)}")
    print("   skeptical read: healthy page, possibly SERP-feature squeeze; a refresh slot spent "
          "here may be wasted - the Week-5 model must learn to demote these.")

# Leakage check on what was written to disk.
out = pd.read_csv(csv_path)
FORBIDDEN_IN_CSV = [c for c in FORBIDDEN if c in out.columns]
print("\nleakage check on CSV:")
print(f"  forbidden columns present: {FORBIDDEN_IN_CSV or 'none'}")
print(f"  reason codes used: {sorted(out['reason_code'].unique())}")
print(f"  action labels used: {sorted(out['action_label'].unique())}")
assert not FORBIDDEN_IN_CSV
print("\nall inputs are trailing-window snapshot fields known at decision time - no future window, "
      "no label-derived column. The queue is decision-support, not prediction.")

top-10 precision: 0.6 (6 of 10 flagged declining)

WEAK #3 content_5c5fab9d41e7: pos 5.4, ctr 0.06%, 1,794 impr/90d, updated 104d ago, label=not declining
   why it looks wrong: no GA4 engagement and/or missing word count - possible measurement gap
   skeptical read: healthy page, possibly SERP-feature squeeze; a refresh slot spent here may be wasted - the Week-5 model must learn to demote these.

WEAK #4 content_f854023b075b: pos 5.1, ctr 0.00%, 627 impr/90d, updated 104d ago, label=not declining
   why it looks wrong: no GA4 engagement and/or missing word count - possible measurement gap; HIGH competition - demand-side squeeze a rewrite may not lift
   skeptical read: healthy page, possibly SERP-feature squeeze; a refresh slot spent here may be wasted - the Week-5 model must learn to demote these.

WEAK #9 content_01618358b525: pos 7.9, ctr 0.09%, 2,110 impr/90d, updated 13d ago, label=not declining
   why it looks wrong: updated within 14 days - may be mid-fix or still ramping; no G

## Self-check

Before you submit, confirm each line honestly:

- [x] Two signal checks with visible bucket tables and n printed — staleness **MIXED** (explained negative, dropped from the rule), CTR-vs-position **CONFIRMED** (flag-linked)
- [x] One rule: transparent score (`search_volume` in pool), ONE reason code (`ranked_but_unclicked`), action label (`refresh_review`)
- [x] Ranked queue written from this notebook: `work/outputs/baseline_action_score.csv`
- [x] Ten reviewed rows with "what would make it wrong" for each
- [x] No future-window or label-derived inputs (guard asserted in §1 and §4)
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, tokens, or private queries anywhere
- [ ] Claims use careful words: observed, measured, directional, decision-support
- [ ] Committed under `work/notebooks/` — then submit the repo URL on the card. Done.